In [ ]:
import smtplib
from email.message import EmailMessage
from datetime import datetime
import os
import subprocess

cir = 's5378'  # 回路名の例
re_learning_count = 3  # 再学習回数の例


# プログラム終了のメール送信
# SMTP簡易送信
msg = EmailMessage() # EmailMessageオブジェクトの作成
msg['subject'] = 'プログラム終了の連絡'  # メールの件名を設定
msg['from'] = os.getenv('MAIL_FROM', 'llinglai70@gmail.com')  # 送信元メールアドレスを設定
msg['to'] = os.getenv('MAIL_TO', 'llinglai70@gmail.com')  # 送信先メールアドレスを設定
msg.set_content('model_sのプログラムが終了しました。\n\n' \
                '回路名：' + cir + '\n' \
                '再学習回数：' + str(re_learning_count) + '\n' \
                '実行プログラム：model_s' + '\n' \
                '日時：' + str(datetime.strftime(datetime.now(), "%Y-%m-%d %H:%M:%S")) + '\n')  # メール本文を設定

# 環境変数から SMTP / msmtp 設定を取得
SMTP_HOST = os.getenv('SMTP_HOST')
SMTP_PORT = int(os.getenv('SMTP_PORT', '587'))
SMTP_USER = os.getenv('SMTP_USER')
SMTP_PASSWORD = os.getenv('SMTP_PASSWORD')
SMTP_USE_SSL = os.getenv('SMTP_USE_SSL', '0').lower() in ('1', 'true', 'yes')
SMTP_TIMEOUT = int(os.getenv('SMTP_TIMEOUT', '30'))

os.environ['SMTP_USER'] = 'llinglai70@gmail.com'
os.environ['SMTP_PASSWORD'] = 'syhjo1-Wommow-xykhif'
os.environ['MAIL_FROM'] = 'llinglai70@gmail.com'

print(SMTP_HOST, SMTP_PORT, SMTP_USER, SMTP_USE_SSL, SMTP_TIMEOUT)

def _log_mail_error(e):
    log_path = os.path.join('.', f'{cir}_mail_error.log')
    with open(log_path, 'a') as lf:
        lf.write(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} mail send failed: {e}\n")
    print("メール送信に失敗しました。詳細は:", log_path)

# msmtp が使えるなら優先して使用し、失敗したら smtplib にフォールバックする
try:
    msmtp_path = None
    try:
        msmtp_path = subprocess.check_output(['which', 'msmtp'], stderr=subprocess.DEVNULL).decode().strip()
    except subprocess.CalledProcessError:
        msmtp_path = None

    if msmtp_path:
        try:
            with subprocess.Popen([msmtp_path, '--read-envelope-from', '-t'], stdin=subprocess.PIPE) as p:
                p.communicate(msg.as_bytes())
            print("メール送信: msmtp を使用しました。")
        except Exception as e_msmtp:
            print("msmtp による送信でエラー:", e_msmtp)
            raise
    else:
        raise FileNotFoundError("msmtp が見つかりません。smtplib を使って送信します。")

except FileNotFoundError:
    # msmtp が無い、または msmtp 呼び出しで例外が発生した場合のフォールバック
    try:
        if SMTP_HOST:
            if SMTP_USE_SSL:
                with smtplib.SMTP_SSL(SMTP_HOST, SMTP_PORT, timeout=SMTP_TIMEOUT) as server:
                    if SMTP_USER and SMTP_PASSWORD:
                        server.login(SMTP_USER, SMTP_PASSWORD)
                    server.send_message(msg)
            else:
                with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=SMTP_TIMEOUT) as server:
                    server.ehlo()
                    server.starttls()
                    server.ehlo()
                    if SMTP_USER and SMTP_PASSWORD:
                        server.login(SMTP_USER, SMTP_PASSWORD)
                    server.send_message(msg)
        else:
            # ローカル MTA を試す（無ければログ）
            try:
                with smtplib.SMTP('localhost', timeout=SMTP_TIMEOUT) as server:
                    server.send_message(msg)
            except Exception as e_local:
                _log_mail_error(e_local)
    except Exception as e:
        _log_mail_error(e)

# stmp_server = os.getenv('SMTP_SERVER')  # SMTPサーバーのアドレスを環境変数から取得 その理由は、コード内に直接記述することでセキュリティリスクが高まるため
# stmp_port = os.getenv('SMTP_PORT', 587)  # SMTPサーバーのポート番号を環境変数から取得。デフォルトは587

# if stmp_server: # 環境変数で外部SMTPが指定されている場合（認証あり）
#     with smtplib.SMTP(stmp_server, stmp_port) as server:
#         server.starttls()
#         smtp_user = os.getenv('SMTP_USER')
#         smtp_password = os.getenv('SMTP_PASSWORD')
#         if smtp_user and smtp_password:
#             server.login(smtp_user, smtp_password)
#         server.send_message(msg)
# else:
#     # sendmail が無い環境向けフォールバック: localhost SMTP を試みる
#     try:
#         with smtplib.SMTP('localhost') as server:
#             server.send_message(msg)
#     except Exception as e:
#         # ログに書き出して処理を続行（管理者権限が無く sendmail/postfix を入れられないケース向け）
#         print("メール送信に失敗しました（sendmail/localhost SMTP 利用不可）。詳細は:")

# # 外部SMTP対応（環境変数で設定）
# # Gmailなどの外部SMTPサーバーを使用する場合、セキュリティ設定やアプリパスワードの設定が必要になることがあります。使用する
# SMTP_HOST = os.getenv('SMTP_HOST')  # SMTPサーバーのホスト名を環境変数から取得
# SMTP_PORT = int(os.getenv('SMTP_PORT', 587))  # SMTPサーバーのポート番号を環境変数から取得。デフォルトは587
# SMTP_USER = os.getenv('SMTP_USER')  # SMTPサーバーのユーザー名を環境変数から取得
# SMTP_PASSWORD = os.getenv('SMTP_PASSWORD')  # SMTPサーバーのパスワードを環境変数から取得
# SMTP_USE_SSL = os.getenv('STMTP_USE_SSL', '0') in ('1', 'true', 'True')  # SSL使用の有無を環境変数から取得
# SMTP_TIMEOUT = int(os.getenv('SMTP_TIMEOUT', '30'))  # タイムアウト時間を環境変数から取得

# def _log_mail_error(e): # メール送信エラーをログに記録する内部関数
#     print("メール送信に失敗しました。詳細は:")

# try:
#     if SMTP_HOST:
#         print("外部SMTPでのメール送信を試みます。")
#         # 外部SMTP指定あり：SSLかSTARTTLSを選択
#         if SMTP_USE_SSL:
#             print("SSLでのメール送信を試みます。")
#             with smtplib.SMTP_SSL(SMTP_HOST, SMTP_PORT, timeout=SMTP_TIMEOUT) as server:
#                 if SMTP_USER and SMTP_PASSWORD:
#                     server.login(SMTP_USER, SMTP_PASSWORD)
#                 server.send_message(msg)
#         else:
#             print("STARTTLSでのメール送信を試みます。")
#             with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=SMTP_TIMEOUT) as server:
#                 server.ehlo()
#                 server.starttls()
#                 server.ehlo()
#                 if SMTP_USER and SMTP_PASSWORD:
#                     server.login(SMTP_USER, SMTP_PASSWORD)
#                 server.send_message(msg)
#     else:
#         # 外部SMTP未指定：localhost を試す（管理者権限無し環境向けフォールバック）
#         try:
#             with smtplib.SMTP('localhost', timeout=SMTP_TIMEOUT) as server:
#                 server.send_message(msg)
#         except Exception as e_local:
#             print("localhost SMTP でのメール送信に失敗しました。外部SMTPを試みます。")
#             _log_mail_error(e_local)
# except Exception as e:
#     print("外部SMTPでのメール送信に失敗しました。")
#     _log_mail_error(e)

print('MAIL_FROM=', os.getenv('MAIL_FROM'))
print('SMTP_USER=', os.getenv('SMTP_USER'))


None 587 None False 30
メール送信: msmtp を使用しました。
MAIL_FROM= llinglai70@gmail.com
SMTP_USER= llinglai70@gmail.com


msmtp: authentication failed (method PLAIN)
msmtp: server message: 535-5.7.8 Username and Password not accepted. For more information, go to
msmtp: server message: 535 5.7.8  https://support.google.com/mail/?p=BadCredentials 98e67ed59e1d1-3476a43ac07sm5054428a91.0 - gsmtp
msmtp: could not send mail (account gmail from /home/ad.dpc/okinaka/.config/msmtp/config)


In [ ]:
export SMTP_USER='your@gmail.com'
export SMTP_PASSWORD='（Google が発行したアプリパスワード）'
export MAIL_FROM='your@gmail.com'   # ノートブックやスクリプトが参照する場合に設定

printf "From: ${MAIL_FROM}\nTo: you@outlook.example\nSubject: test\n\nhello\n" \
  | msmtp --file /home/ad.dpc/okinaka/.config/msmtp/config --read-envelope-from -t --debug

In [19]:
import os
print(repr(os.getenv('SMTP_HOST')))
print(os.environ.get('SMTP_HOST'))  # None なら未設定

None
None


In [8]:
import smtplib
from email.message import EmailMessage
msg = EmailMessage()
msg['From'] = 'm806002y@mails.cc.ehime-u.ac.jp'
msg['To'] = 'm806002y@mails.cc.ehime-u.ac.jp'
msg['Subject'] = 'test'
msg.set_content('test')
s = smtplib.SMTP('localhost', 25, timeout=10)
s.set_debuglevel(1)   # SMTP 通信ログを表示
try:
    s.send_message(msg)
except Exception as e:
    print('send failed:', e)
finally:
    s.quit()

OSError: [Errno 101] Network is unreachable